In [ ]:
import os
import shutil

base_dir = "/content/tiny-imagenet-200"
val_dir = os.path.join(base_dir, "val")
images_dir = os.path.join(val_dir, "images")
ann_file = os.path.join(val_dir, "val_annotations.txt")

# Read annotations
with open(ann_file) as f:
    annotations = [line.strip().split('\t') for line in f]

# Create class folders and move images
for img, cls, *_ in annotations:
    cls_dir = os.path.join(val_dir, cls)
    os.makedirs(cls_dir, exist_ok=True)
    shutil.move(
        os.path.join(images_dir, img),
        os.path.join(cls_dir, img)
    )

os.rmdir(images_dir)

In [ ]:
!pip install -q tf-models-official
!pip install transformers

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

!pip -q install tf-keras

In [ ]:
import tensorflow as tf
import tensorflow.keras.layers as tfla
import tensorflow.keras.models as tfm
import tensorflow.keras.optimizers as tfo
import tensorflow.keras.losses as tflo
import matplotlib.pyplot as plt
from official.vision.ops import augment

In [ ]:
with open("tiny-imagenet-200/wnids.txt") as f:
    wnids = [line.strip() for line in f]

train_ds = tf.keras.utils.image_dataset_from_directory(
    "tiny-imagenet-200/train",
    labels="inferred",
    label_mode="categorical",
    class_names=wnids,
    image_size=(256, 256),
    batch_size=None,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "tiny-imagenet-200/val",
    labels="inferred",
    label_mode="categorical",
    class_names=wnids,
    image_size=(224, 224),
    batch_size=None,
)

Found 100000 files belonging to 200 classes.
Found 10000 files belonging to 200 classes.


In [ ]:
def random_crop(image, label):
  # image shape: [hei, wid, cha]
  # label shape: [num_classes]
  image = tf.image.random_crop(image, (224, 224, 3))
  image = tf.image.random_flip_left_right(image)
  return image, label

In [ ]:
train_ds = train_ds.map(random_crop)

In [ ]:
def apply_randaugment(image):
  # image shape: [hei, wid, cha]
  augmenter = augment.RandAugment(num_layers=2, magnitude=9)
  return augmenter.distort(image)

In [ ]:
train_ds = train_ds.map(lambda x, y : (apply_randaugment(x), y), num_parallel_calls=tf.data.AUTOTUNE)

In [ ]:
def color_jitter(image, label):
  image = tf.image.random_brightness(image, 0.2)
  image = tf.image.random_contrast(image, 0.8, 1.2)
  image = tf.image.random_saturation(image, 0.8, 1.2)
  image = tf.image.random_hue(image, 0.02)
  image = tf.clip_by_value(image, 0, 255)
  return image, label

In [ ]:
!pip install --upgrade keras-cv

import keras_cv

blur_layer = keras_cv.layers.RandomGaussianBlur(
    kernel_size=3,
    factor=(0.1, 2.0)
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 650.7/650.7 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 950.8/950.8 kB 41.3 MB/s eta 0:00:00
Using TensorFlow backend


In [ ]:
def apply_jitter_blur(image, label):
  if(tf.random.uniform(shape=()) < 0.2):
    image, _ = color_jitter(image, label)

  if(tf.random.uniform(shape=()) < 0.1):
    image = blur_layer(image)

  return(image, label)

In [ ]:
# you can apply jitter blur here, but we align with the original paper
# train_ds = train_ds.map(apply_jitter_blur, num_parallel_calls=tf.data.AUTOTUNE)

In [ ]:
def scaling(image, label):
  # image shape: [hei, wid, cha]
  # label shape: [num_classes]
  mean = tf.constant([0.485, 0.456, 0.406])
  std = tf.constant([0.229, 0.224, 0.225])

  image = (image / 255.0 - mean) / std
  return image, label

In [ ]:
train_ds = train_ds.map(scaling, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.map(scaling, num_parallel_calls=tf.data.AUTOTUNE)

In [ ]:
train_ds = train_ds.batch(64)

In [ ]:
def mixup(images, labels):
  # images shape: [batchsize, hei, wid, cha]
  # labels shape: [batchsize, num_classes]
  batch_size = tf.shape(images)[0]

  # this makes image mostly all image A or all image B
  gamma_1 = tf.random.gamma(shape=(batch_size, 1), alpha=0.2)
  gamma_2 = tf.random.gamma(shape=(batch_size, 1), alpha=0.2)
  lam = gamma_1 / (gamma_1 + gamma_2)

  indices = tf.random.shuffle(tf.range(batch_size))
  shuffled_images = tf.gather(images, indices)
  shuffled_labels = tf.gather(labels, indices)

  images_lam = tf.reshape(lam, [-1, 1, 1, 1])
  labels_lam = tf.reshape(lam, [-1, 1])

  mixed_images = images_lam * images + (1 - images_lam) * shuffled_images
  mixed_labels = labels_lam * labels + (1 - labels_lam) * shuffled_labels

  return(mixed_images, mixed_labels)

In [ ]:
mixedup_train_ds = train_ds.map(mixup, num_parallel_calls=tf.data.AUTOTUNE)
combined_ds = train_ds.concatenate(mixedup_train_ds)

for images, _ in combined_ds.take(1):
  print(tf.reduce_mean(tf.abs(images[0])))

tf.Tensor(1.3217051, shape=(), dtype=float32)


In [ ]:
def cutmix(images, labels):
  # images shape: [batchsize, hei, wid, cha]
  # labels shape: [batchsize, num_classes]
  batch_size = tf.shape(images)[0]
  img_height = tf.shape(images)[1]
  img_width = tf.shape(images)[2]

  # this makes image mostly all image A or all image B
  g1 = tf.random.gamma(shape=(batch_size, 1, 1), alpha=0.2)
  g2 = tf.random.gamma(shape=(batch_size, 1, 1), alpha=0.2)
  lam = g1 / (g1 + g2)
  # lam shape: [batchsize, 1, 1]

  cut_w = tf.cast(tf.cast(img_width, tf.float32) * tf.sqrt(1 - lam), tf.int32)
  cut_h = tf.cast(tf.cast(img_height, tf.float32) * tf.sqrt(1 - lam), tf.int32)
  # cutw, cuth shape: [batchsize, 1, 1]

  random_x = tf.random.uniform(shape=(batch_size, 1, 1), minval=0, maxval=1.)
  random_y = tf.random.uniform(shape=(batch_size, 1, 1), minval=0, maxval=1.)

  x1 = tf.cast(img_width - cut_w - 1, tf.float32)
  y1 = tf.cast(img_height - cut_h - 1, tf.float32)

  cx = tf.cast(cut_w // 2, tf.int32) + tf.cast(random_x * x1, tf.int32)
  cy = tf.cast(cut_h // 2, tf.int32) + tf.cast(random_y * y1, tf.int32)

  x1 = tf.cast(cx - cut_w // 2, tf.int32)
  x2 = tf.cast(cx + cut_w // 2, tf.int32)
  y1 = tf.cast(cy - cut_h // 2, tf.int32)
  y2 = tf.cast(cy + cut_h // 2, tf.int32)

  x_indices = tf.range(img_width)
  y_indices = tf.range(img_height)
  x_grid, y_grid = tf.meshgrid(x_indices, y_indices)
  x_grid = tf.reshape(x_grid, [1, img_height, img_width])
  y_grid = tf.reshape(y_grid, [1, img_height, img_width])

  mask = tf.logical_and(tf.logical_and(x_grid >= x1, x_grid < x2), tf.logical_and(
      y_grid >= y1, y_grid < y2))

  mask = tf.cast(mask, tf.float32)
  mask = tf.expand_dims(mask, -1)

  shuffled_indices = tf.random.shuffle(tf.range(batch_size))
  shuffled_images = tf.gather(images, shuffled_indices)
  shuffled_labels = tf.gather(labels, shuffled_indices)

  cutmix_images = (1 - mask) * images + mask * shuffled_images

  lam_labels = tf.cast(cut_w * cut_h, tf.float32) / tf.cast(img_width * img_height, tf.float32)
  cutmix_labels = (1 - tf.squeeze(lam_labels, axis=1)) * labels + tf.squeeze(lam_labels, axis=1) * shuffled_labels

  return cutmix_images, cutmix_labels

In [ ]:
cutmix_train_ds = train_ds.map(cutmix, num_parallel_calls=tf.data.AUTOTUNE)
combined_ds = train_ds.concatenate(cutmix_train_ds)

In [ ]:
def erasing(images, labels):
  # images shape: [batchsize, hei, wid, cha]
  # labels shape: [batchsize, num_classes]
  batch_size = tf.shape(images)[0]
  img_height = tf.shape(images)[1]
  img_width = tf.shape(images)[2]

  num = tf.random.uniform(shape=(batch_size, 1, 1), minval=0.02, maxval=0.33,
                          dtype=tf.float32)

  cut_w = tf.cast(tf.cast(img_width, tf.float32) * tf.sqrt(num), tf.int32)
  cut_h = tf.cast(tf.cast(img_height, tf.float32) * tf.sqrt(num), tf.int32)

  random_x = tf.random.uniform(shape=(batch_size, 1, 1), minval=0, maxval=1.)
  random_y = tf.random.uniform(shape=(batch_size, 1, 1), minval=0, maxval=1.)

  x1 = tf.cast(img_width - cut_w - 1, tf.float32)
  y1 = tf.cast(img_height - cut_h - 1, tf.float32)

  cx = tf.cast(cut_w // 2, tf.int32) + tf.cast(random_x * x1, tf.int32)
  cy = tf.cast(cut_h // 2, tf.int32) + tf.cast(random_y * y1, tf.int32)

  x1 = tf.cast(cx - cut_w // 2, tf.int32)
  x2 = tf.cast(cx + cut_w // 2, tf.int32)
  y1 = tf.cast(cy - cut_h // 2, tf.int32)
  y2 = tf.cast(cy + cut_h // 2, tf.int32)

  x_indices = tf.range(img_width)
  y_indices = tf.range(img_height)
  x_grid, y_grid = tf.meshgrid(x_indices, y_indices)
  x_grid = tf.reshape(x_grid, [1, img_height, img_width])
  y_grid = tf.reshape(y_grid, [1, img_height, img_width])

  mask = tf.logical_and(tf.logical_and(x_grid >= x1, x_grid < x2), tf.logical_and(
      y_grid >= y1, y_grid < y2))

  mask = tf.cast(mask, tf.float32)
  mask = tf.expand_dims(mask, -1)

  erase_images = (1 - mask) * images

  return(erase_images, labels)

In [ ]:
def convert_back(image):
  mean = tf.constant([0.485, 0.456, 0.406])
  std = tf.constant([0.229, 0.224, 0.225])

  image = (image * std + mean) * 255
  image = tf.clip_by_value(image, 0, 255)
  image = tf.cast(image, tf.int32)

  return image

In [ ]:
def load_tinyimagenet_label_maps(tiny_imagenet_root):
    # Load wnids (index -> wnid)
    with open(f"{tiny_imagenet_root}/wnids.txt") as f:
        wnids = [line.strip() for line in f]

    # Load wnid -> words
    wnid_to_words = {}
    with open(f"{tiny_imagenet_root}/words.txt") as f:
        for line in f:
            wnid, words = line.strip().split("\t")
            wnid_to_words[wnid] = words

    return wnids, wnid_to_words

In [ ]:
wnids, wnid_to_words = load_tinyimagenet_label_maps("tiny-imagenet-200")

In [ ]:
def one_hot_to_tinyimagenet_word(one_hot_label, wnids, wnid_to_words):
    """
    one_hot_label: tf.Tensor or np.array, shape (200,)
    """
    # Convert one-hot to index
    if isinstance(one_hot_label, tf.Tensor):
        label_idx = int(tf.argmax(one_hot_label).numpy())
    else:
        label_idx = int(one_hot_label.argmax())

    wnid = wnids[label_idx]
    return wnid_to_words[wnid]

In [ ]:
#%matplotlib inline
#
#for images, labels in train_ds.take(1):
#  imagess, labelss = erasing(images, labels)
#  print(one_hot_to_tinyimagenet_word(labelss[0], wnids, wnid_to_words))
#  plt.imshow(convert_back(imagess[0]).numpy())
#  plt.axis("off")
#  plt.show()

In [ ]:
erase_train_ds = train_ds.map(erasing)
combined_ds = train_ds.concatenate(erase_train_ds)

In [ ]:
def label_smoothing(images, labels, epsilon=0.1):
  num_classes = tf.cast(200, tf.float32)
  labels = labels * (1 - epsilon) + epsilon / num_classes
  return images, labels

In [ ]:
combined_ds = combined_ds.map(label_smoothing, num_parallel_calls=tf.data.AUTOTUNE)

In [ ]:
combined_ds = combined_ds.shuffle(buffer_size=100)

In [ ]:
# add pos_token and cls_token to the given input
class embeddings(tfla.Layer):
  def __init__(self):
    super(embeddings, self).__init__()
    # define pos_emb with shape [1, D, N]
    self.pos_emb = self.add_weight(
        shape=(1, 198, 384),
        initializer="zeros",
        trainable=True
    )

    # define cls_token with shape [1, 1, N]
    self.cls_emb = self.add_weight(
        shape=(1, 1, 384),
        initializer="zeros",
        trainable=True
    )

    # define dis_token with shape [1, 1, N]
    self.dis_emb = self.add_weight(
        shape=(1, 1, 384),
        initializer="zeros",
        trainable=True
    )

  # input with shape [B, D, N]
  def call(self, inputs, training=False):
    batchSize = tf.shape(inputs)[0]
    # tile cls token to [B, 1, N] then concat
    cls_token = tf.tile(self.cls_emb, [batchSize, 1, 1])
    dis_token = tf.tile(self.dis_emb, [batchSize, 1, 1])
    x = tf.concat([cls_token, dis_token, inputs], axis = 1)
    x = x + self.pos_emb
    return(x)

In [ ]:
# encoder block with 12 heads
class encoder(tfla.Layer):
  def __init__(self):
    super(encoder, self).__init__()
    # multihead attention num_heads is 12
    self.mha = tfla.MultiHeadAttention(
        num_heads=12,
        key_dim=384 // 12,
        dropout=0.0
    )

    self.layernorm1 = tfla.LayerNormalization()
    self.layernorm2 = tfla.LayerNormalization()

    self.dense1 = tfla.Dense(
        384*4,
        activation="gelu"
    )
    self.dense2 = tfla.Dense(
        384
    )

  def dropLayers(self, inputs, training=False):
    if(not training):
      return(inputs)

    mask = tf.random.uniform(shape=(tf.shape(inputs)[0], 1, 1), minval=0, maxval=1)
    mask = tf.floor(mask+0.9)

    x = inputs * tf.cast(mask, tf.float32) / 0.9

    return(x)

  def call(self, inputs, training=False):
    x = self.layernorm1(inputs)
    x = self.mha(x, x)

    x = self.dropLayers(x, training=training)
    x = x + inputs

    x1 = x
    x = self.layernorm2(x)
    x = self.dense1(x)
    x = self.dense2(x)

    x = self.dropLayers(x, training=training)
    x = x + x1

    return x

In [ ]:
inputs = tfla.Input(shape=(224, 224, 3))

x = tfla.Conv2D(
    384,
    16,
    strides=(16, 16),
    padding="valid"
)(inputs)

x = tfla.Reshape((-1, 384))(x)
x = embeddings()(x)

for _ in range(12):
  x = encoder()(x)

cls_token = x[:, 0, :]
dis_token = x[:, 1, :]

cls_output = tfla.Dense(200, activation="softmax")(cls_token)
dis_output = tfla.Dense(200, activation="softmax")(dis_token)

model = tfm.Model(inputs=inputs, outputs=[cls_output, dis_output])

In [ ]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 conv2d (Conv2D)             (None, 14, 14, 384)          295296    ['input_1[0][0]']             
                                                                                                  
 reshape (Reshape)           (None, 196, 384)             0         ['conv2d[0][0]']              
                                                                                                  
 embeddings (embeddings)     (None, 198, 384)             76800     ['reshape[0][0]']             
                                                                                              

In [ ]:
def convert_back(image):
  mean = tf.constant([0.485, 0.456, 0.406])
  std = tf.constant([0.229, 0.224, 0.225])

  image = (image * std + mean) * 255
  image = tf.clip_by_value(image, 0, 255)

  mean = tf.constant([0.5, 0.5, 0.5])
  std = tf.constant([0.5, 0.5, 0.5])

  image = (image / 255.0 - mean) / std

  return image

In [ ]:
def tonchw(image):
  return tf.transpose(image, [0, 3, 1, 2])

In [ ]:
ft_train_ds = train_ds.map(lambda x, y: (convert_back(x), y))
ft_train_ds = ft_train_ds.map(lambda x, y: (tonchw(x), y))

In [ ]:
from transformers import TFViTModel

teacher = TFViTModel.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    from_pt=True,
    use_safetensors=False
)

inputs = tfla.Input(shape=(3, 224, 224))

outputs = teacher(inputs, training=True)
cls_token = outputs.last_hidden_state[:, 0, :]
x = tfla.Dropout(0.1)(cls_token)
x = tfla.Dense(200, activation="softmax")(x)

teacher_model = tf.keras.Model(inputs, x)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/346M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
All PyTorch model weights were used when initializing TFViTModel.

All the weights of TFViTModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFViTModel for predictions without further training.


In [ ]:
teacher.trainable = False

teacher_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
    metrics=["accuracy"]
)

teacher_model.fit(ft_train_ds, epochs=5)

Epoch 1/5
1563/1563 [==============================] - 227s 137ms/step - loss: 3.1499 - accuracy: 0.4267
Epoch 2/5
1563/1563 [==============================] - 214s 137ms/step - loss: 2.3363 - accuracy: 0.4953
Epoch 3/5
1563/1563 [==============================] - 213s 136ms/step - loss: 2.2102 - accuracy: 0.5102
Epoch 4/5
1563/1563 [==============================] - 214s 137ms/step - loss: 2.1448 - accuracy: 0.5210
Epoch 5/5
1563/1563 [==============================] - 214s 137ms/step - loss: 2.0987 - accuracy: 0.5302


In [ ]:
teacher.trainable = True

teacher_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
    metrics=["accuracy"]
)

teacher_model.fit(ft_train_ds, epochs=10)

Epoch 1/10


1563/1563 [==============================] - 501s 287ms/step - loss: 1.3845 - accuracy: 0.6716
Epoch 2/10
1563/1563 [==============================] - 449s 287ms/step - loss: 1.1225 - accuracy: 0.7239
Epoch 3/10
1563/1563 [==============================] - 448s 286ms/step - loss: 1.0150 - accuracy: 0.7464
Epoch 4/10
1563/1563 [==============================] - 449s 287ms/step - loss: 0.9396 - accuracy: 0.7639
Epoch 5/10
1563/1563 [==============================] - 449s 287ms/step - loss: 0.8878 - accuracy: 0.7753
Epoch 6/10
1563/1563 [==============================] - 449s 287ms/step - loss: 0.8370 - accuracy: 0.7868
Epoch 7/10
1563/1563 [==============================] - 449s 287ms/step - loss: 0.7954 - accuracy: 0.7958
Epoch 8/10
1563/1563 [==============================] - 449s 287ms/step - loss: 0.7692 - accuracy: 0.8020
Epoch 9/10
1563/1563 [==============================] - 449s 287ms/step - loss: 0.7337 - accuracy: 0.8112
Epoch 10/10
1563/1563 [==============================] - 

In [ ]:
def get_teacher_labels(images):
  images = convert_back(images)
  images = tonchw(images)
  outputs = teacher_model(images, training=False)

  return outputs

In [ ]:
class WarmupCosine(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_lr, warmup_steps, total_steps, alpha=0.1):
        super().__init__()
        self.base_lr = base_lr
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps

        self.cosine = tf.keras.optimizers.schedules.CosineDecay(
            initial_learning_rate=base_lr,
            decay_steps=total_steps - warmup_steps,
            alpha=alpha,
        )

    def __call__(self, step):
        step = tf.cast(step, tf.float32)

        return tf.cond(
            step < self.warmup_steps,
            lambda: self.base_lr * step / self.warmup_steps,
            lambda: self.cosine(step - self.warmup_steps),
        )

    def get_config(self):
        return {
            "base_lr": self.base_lr,
            "warmup_steps": self.warmup_steps,
            "total_steps": self.total_steps,
        }

lr_schedule = WarmupCosine(
    base_lr=1e-5,
    warmup_steps=10_000,
    total_steps=100_000,
    alpha=0.1,
)

optimizer = tfo.AdamW(
    learning_rate=lr_schedule,
    weight_decay=0.05,
)

loss_fn = tflo.CategoricalCrossentropy(from_logits=False)

In [ ]:
@tf.function
def train_step(images, labels):
  teacher_labels = get_teacher_labels(images)

  with tf.GradientTape() as tape:
    cls_outputs, dis_outputs = model(images, training=True)

    cls_loss = loss_fn(labels, cls_outputs)
    dis_loss = loss_fn(teacher_labels, dis_outputs)

    total_loss = 0.5 * cls_loss + 0.5 * dis_loss

  gradients = tape.gradient(total_loss, model.trainable_variables)
  optimizer.apply_gradients(zip(gradients, model.trainable_variables))

  return total_loss, cls_loss, dis_loss

In [ ]:
for epoch in range(40):

  print(f"Start of epoch {epoch}\n")

  total_step_one_epoch = 0.0
  total_loss_one_epoch = 0.0
  for step, (images, labels) in enumerate(combined_ds):
    total_loss, cls_loss, dis_loss = train_step(images, labels)
    if(step % 50 == 0):
      print(f"total_loss: {total_loss}, cls_loss: {cls_loss}, dis_loss: {dis_loss}\n")
      total_step_one_epoch += 1.0
      total_loss_one_epoch += total_loss.numpy()

  print(f"average_loss_for epoch {epoch}: {total_loss_one_epoch / total_step_one_epoch}\n")

Streaming output truncated to the last 5000 lines.
total_loss: 5.400774955749512, cls_loss: 5.481362342834473, dis_loss: 5.320187568664551

total_loss: 5.40461540222168, cls_loss: 5.384621620178223, dis_loss: 5.4246087074279785

total_loss: 5.37712287902832, cls_loss: 5.352841854095459, dis_loss: 5.40140438079834

total_loss: 5.413539886474609, cls_loss: 5.469906330108643, dis_loss: 5.357173919677734

total_loss: 5.154181957244873, cls_loss: 5.1937055587768555, dis_loss: 5.114658355712891

total_loss: 5.471506595611572, cls_loss: 5.43252420425415, dis_loss: 5.510488986968994

total_loss: 5.48081111907959, cls_loss: 5.536480903625488, dis_loss: 5.425141334533691

total_loss: 5.246059894561768, cls_loss: 5.160480499267578, dis_loss: 5.331639289855957

total_loss: 5.345541000366211, cls_loss: 5.3845672607421875, dis_loss: 5.306514263153076

total_loss: 5.308314323425293, cls_loss: 5.33726692199707, dis_loss: 5.279362201690674

total_loss: 5.33698844909668, cls_loss: 5.287724494934082, dis

In [ ]:
model.save_weights("deit_tinyimagenet.weights.h5")
from google.colab import files
files.download("deit_tinyimagenet.weights.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>